# 02 · Feature selection + scaling (train-only)
Ranks features with Random Forest importance + Mutual Information on a class-capped **training** sample, picks *k* on the **validation day** (attack-vs-benign PR-AUC), then fits a signed-log1p + StandardScaler on train and writes model-ready arrays to `processed/arrays/`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/NIDS_CICIDS2017_Project')
RAW, EXTRACT, ART, PROC, REP, MODELS = [ROOT/d for d in ['raw','extracted_generated_flows','artifacts','processed','reports','models']]
for d in [RAW, EXTRACT, ART, PROC, REP, MODELS]: d.mkdir(parents=True, exist_ok=True)
SEED = 42

import json, joblib
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
rng = np.random.default_rng(SEED)

meta = json.loads((ART/'preprocessing_meta.json').read_text()); lab = json.loads((ART/'label_mapping.json').read_text())
feats, known, glob = meta['features'], lab['known'], lab['global']
kidx = {l: i for i, l in enumerate(known)}
tr, va, te = [pd.read_parquet(PROC/f'{s}.parquet') for s in ['train','val','test']]
ytr = tr['Label'].map(kidx).to_numpy()
print(len(feats), 'features |', len(known), 'known classes')

Mounted at /content/drive
44 features | 8 known classes


In [2]:
def capped_idx(y, cap):
    return np.sort(np.concatenate([rng.choice(np.where(y == c)[0], min(cap, (y == c).sum()), replace=False) for c in np.unique(y)]))

Xtr_all = tr[feats].to_numpy('float32')
i_rf = capped_idx(ytr, 40_000); i_mi = capped_idx(ytr, 8_000)

rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, class_weight='balanced_subsample', min_samples_leaf=2, random_state=SEED)
rf.fit(Xtr_all[i_rf], ytr[i_rf])
mi = mutual_info_classif(Xtr_all[i_mi], ytr[i_mi], random_state=SEED)

rank = pd.DataFrame({'feature': feats, 'rf_importance': rf.feature_importances_, 'mutual_info': mi})
rank['combined_rank'] = (rank['rf_importance'].rank(ascending=False) + rank['mutual_info'].rank(ascending=False)) / 2
rank = rank.sort_values(['combined_rank', 'rf_importance'], ascending=[True, False]).reset_index(drop=True)
rank.to_csv(REP/'feature_ranking.csv', index=False); display(rank.head(20))

,feature,rf_importance,mutual_info,combined_rank
0,Flow IAT Mean,0.040112,1.563724,4.5
1,Packet Length Mean,0.055227,1.391249,6.0
2,Bwd Packets/s,0.050323,1.413880,6.0
3,Packet Length Variance,0.048495,1.464219,6.0
4,Fwd Header Length,0.060244,1.208593,8.0
5,Fwd Packet Length Max,0.041994,1.328106,8.5
6,Flow Packets/s,0.029751,1.560584,9.0
7,Bwd Packet Length Max,0.072273,1.168268,9.5
8,Flow Duration,0.032163,1.495199,10.0
9,Fwd IAT Mean,0.032140,1.507886,10.0


In [3]:
# Choose k on the validation day (Thursday): attack-vs-benign average precision of a quick RF using the top-k features.
n = len(feats)
cands = sorted({k for k in [10, 20, 26, 30, 40, 48] if k < n} | {n})
vi = np.sort(rng.choice(len(va), min(150_000, len(va)), replace=False))
Xva = va[feats].iloc[vi]; yva_bin = (va['Label'].iloc[vi] != 'BENIGN').astype(int).to_numpy()
res = []
for k in cands:
    top = rank['feature'][:k].tolist()
    idx_fit = [feats.index(f) for f in top]
    clf = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, class_weight='balanced_subsample', random_state=SEED)
    clf.fit(Xtr_all[i_rf][:, idx_fit], ytr[i_rf])
    s = 1 - clf.predict_proba(Xva[top].to_numpy('float32'))[:, 0]
    res.append({'k': k, 'val_AP': average_precision_score(yva_bin, s), 'val_ROC_AUC': roc_auc_score(yva_bin, s)})
    print(res[-1])
kres = pd.DataFrame(res); kres.to_csv(REP/'k_selection.csv', index=False)
best = kres['val_AP'].max()
K_SEL = int(kres[kres['val_AP'] >= best - 0.005]['k'].min())      # smallest k within 0.005 AP of the best
selected = rank['feature'][:K_SEL].tolist()
print('Selected k =', K_SEL)
(ART/'selected_features.json').write_text(json.dumps({'k': K_SEL, 'features': selected, 'k_results': res}, indent=2))

{'k': 10, 'val_AP': np.float64(0.006747067521655255), 'val_ROC_AUC': np.float64(0.4857813538925034)}
{'k': 20, 'val_AP': np.float64(0.016257577872109692), 'val_ROC_AUC': np.float64(0.7791309886252342)}
{'k': 26, 'val_AP': np.float64(0.010579780764233288), 'val_ROC_AUC': np.float64(0.5548658610669571)}
{'k': 30, 'val_AP': np.float64(0.28156172287083286), 'val_ROC_AUC': np.float64(0.9801334988300728)}
{'k': 40, 'val_AP': np.float64(0.3422299899588132), 'val_ROC_AUC': np.float64(0.9791540612269719)}
{'k': 44, 'val_AP': np.float64(0.29611946870940886), 'val_ROC_AUC': np.float64(0.9751551200655019)}
Selected k = 40


1614

In [4]:
# Scaling: signed log1p -> StandardScaler (fit on train) -> clip to [-5, 5]. Then save arrays.
def slog(a): return (np.sign(a) * np.log1p(np.abs(a))).astype('float32')
scaler = StandardScaler().fit(slog(tr[selected].to_numpy('float32')))
joblib.dump(scaler, ART/'scaler.joblib')
OUT = PROC/'arrays'; OUT.mkdir(exist_ok=True)
for name, df in [('train', tr), ('val', va), ('test', te)]:
    X = np.clip(scaler.transform(slog(df[selected].to_numpy('float32'))), -5, 5).astype('float32')
    np.save(OUT/f'X_{name}.npy', X)
    np.save(OUT/f'y_known_{name}.npy', df['Label'].map(kidx).fillna(-1).astype('int8').to_numpy())   # -1 = unseen attack family
    np.save(OUT/f'y_bin_{name}.npy', (df['Label'] != 'BENIGN').astype('int8').to_numpy())
    np.save(OUT/f'y_global_{name}.npy', df['Label'].map(glob).astype('int16').to_numpy())
    np.save(OUT/f'day_id_{name}.npy', df['day_id'].to_numpy('int8'))
    print(name, X.shape)

train (1398398, 40)
val (458968, 40)
test (703245, 40)
